# tarea 1 - análisis de textos

Voy a comparar *Alice's Adventures in Wonderland* y *The Wonderful Wizard of Oz*. Los dos son libros de fantasía en inglés y se pueden descargar completos

Fuentes que usé:

- [Alice en Project Gutenberg](https://www.gutenberg.org/ebooks/11) y su [texto en txt](https://www.gutenberg.org/files/11/11-0.txt)
- [Oz en Project Gutenberg](https://www.gutenberg.org/ebooks/55) y su [texto en txt](https://www.gutenberg.org/files/55/55-0.txt)

Voy a revisar palabras, frecuencias, n-gramas, puntuación y emojis

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from urllib.request import Request, urlopen
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

## textos

Los bajo en txt y quito lo que Gutenberg pone al inicio y al final

In [ ]:
enlace_alicia = 'https://www.gutenberg.org/files/11/11-0.txt'
enlace_oz = 'https://www.gutenberg.org/files/55/55-0.txt'

# aquí bajo los textos para no depender de archivos guardados en mi equipo
solicitud = Request(enlace_alicia, headers={'User-Agent': 'Mozilla/5.0'})
texto_alicia = urlopen(solicitud, timeout=30).read().decode('utf-8-sig')
solicitud = Request(enlace_oz, headers={'User-Agent': 'Mozilla/5.0'})
texto_oz = urlopen(solicitud, timeout=30).read().decode('utf-8-sig')

inicio = re.search(r'\*\*\* START OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*', texto_alicia, re.IGNORECASE)
fin = re.search(r'\*\*\* END OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*', texto_alicia, re.IGNORECASE)
texto_alicia = texto_alicia[inicio.end():fin.start()].strip()

inicio = re.search(r'\*\*\* START OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*', texto_oz, re.IGNORECASE)
fin = re.search(r'\*\*\* END OF (?:THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*', texto_oz, re.IGNORECASE)
texto_oz = texto_oz[inicio.end():fin.start()].strip()

# alice va primero y oz después
len(texto_alicia), len(texto_oz)

## limpieza

Paso todo a minúsculas y quito palabras muy comunes. También saco algunas palabras que vienen de la edición

In [ ]:
patron_palabras = r"[a-z]+(?:['’][a-z]+)?"
palabras_alicia = re.findall(patron_palabras, texto_alicia.lower())
palabras_oz = re.findall(patron_palabras, texto_oz.lower())

palabras_alicia = [palabra.replace('’', "'") for palabra in palabras_alicia]
palabras_oz = [palabra.replace('’', "'") for palabra in palabras_oz]

palabras_vacias = set(ENGLISH_STOP_WORDS)
palabras_extra = {'chapter', 'illustration', 'illustrations'}
palabras_vacias.update(palabras_extra)

terminos_alicia = [palabra for palabra in palabras_alicia if palabra not in palabras_vacias and len(palabra) > 1]
terminos_oz = [palabra for palabra in palabras_oz if palabra not in palabras_vacias and len(palabra) > 1]

# reviso cuántas quedaron en alice y en oz
len(terminos_alicia), len(terminos_oz)

## datos generales

Aquí saco algunas cantidades de los dos libros. La riqueza léxica es la división entre palabras únicas y palabras analizadas

In [ ]:
oraciones_alicia = len([parte for parte in re.split(r'[.!?]+', texto_alicia) if parte.strip()])
oraciones_oz = len([parte for parte in re.split(r'[.!?]+', texto_oz) if parte.strip()])
parrafos_alicia = len([parte for parte in re.split(r'\n\s*\n', texto_alicia) if parte.strip()])
parrafos_oz = len([parte for parte in re.split(r'\n\s*\n', texto_oz) if parte.strip()])

largo_palabras_alicia = [len(palabra.replace("'", '')) for palabra in palabras_alicia]
largo_palabras_oz = [len(palabra.replace("'", '')) for palabra in palabras_oz]
riqueza_alicia = round(len(set(terminos_alicia)) / len(terminos_alicia), 4)
riqueza_oz = round(len(set(terminos_oz)) / len(terminos_oz), 4)
media_alicia = round(float(np.mean(largo_palabras_alicia)), 2)
media_oz = round(float(np.mean(largo_palabras_oz)), 2)

tabla_resumen = pd.DataFrame([
    ['alice', len(texto_alicia), len(palabras_alicia), oraciones_alicia, parrafos_alicia, len(terminos_alicia),
     len(set(terminos_alicia)), sum(valor == 1 for valor in Counter(terminos_alicia).values()), riqueza_alicia,
     media_alicia, round(float(np.median(largo_palabras_alicia)), 2)],
    ['oz', len(texto_oz), len(palabras_oz), oraciones_oz, parrafos_oz, len(terminos_oz),
     len(set(terminos_oz)), sum(valor == 1 for valor in Counter(terminos_oz).values()), riqueza_oz,
     media_oz, round(float(np.median(largo_palabras_oz)), 2)]
], columns=['libro', 'caracteres', 'palabras', 'oraciones', 'parrafos', 'tokens', 'unicas', 'hapax',
            'riqueza lexica', 'longitud media', 'longitud mediana'])
tabla_resumen

## palabras más usadas

Tomo las 15 que más aparecen. En la gráfica uso la cantidad por cada 10 mil palabras porque los libros no miden lo mismo

In [ ]:
frecuencias_alicia = Counter(terminos_alicia)
frecuencias_oz = Counter(terminos_oz)
frecuentes_alicia = frecuencias_alicia.most_common(15)
frecuentes_oz = frecuencias_oz.most_common(15)

tabla_frecuencias = pd.DataFrame({
    'palabra alice': [dato[0] for dato in frecuentes_alicia],
    'conteo alice': [dato[1] for dato in frecuentes_alicia],
    'palabra oz': [dato[0] for dato in frecuentes_oz],
    'conteo oz': [dato[1] for dato in frecuentes_oz]
})
tabla_frecuencias

In [ ]:
palabras_grafica = [dato[0] for dato in frecuentes_alicia][::-1]
tasas_grafica = [dato[1] / len(terminos_alicia) * 10000 for dato in frecuentes_alicia][::-1]
plt.figure(figsize=(8, 5))
plt.barh(palabras_grafica, tasas_grafica)
plt.title('Palabras frecuentes en Alice')
plt.xlabel('apariciones por cada 10,000 tokens')
plt.show()

palabras_grafica = [dato[0] for dato in frecuentes_oz][::-1]
tasas_grafica = [dato[1] / len(terminos_oz) * 10000 for dato in frecuentes_oz][::-1]
plt.figure(figsize=(8, 5))
plt.barh(palabras_grafica, tasas_grafica)
plt.title('Palabras frecuentes en Oz')
plt.xlabel('apariciones por cada 10,000 tokens')
plt.show()

## distribuciones

Ahora comparo el largo de las palabras y qué tan rápido bajan sus frecuencias

In [ ]:
longitudes_alicia = [len(palabra.replace("'", '')) for palabra in terminos_alicia]
longitudes_oz = [len(palabra.replace("'", '')) for palabra in terminos_oz]
orden_alicia = sorted([valor / len(terminos_alicia) for valor in frecuencias_alicia.values()], reverse=True)
orden_oz = sorted([valor / len(terminos_oz) for valor in frecuencias_oz.values()], reverse=True)

longitud_maxima = max(max(longitudes_alicia), max(longitudes_oz))
intervalos = range(1, longitud_maxima + 2)
plt.hist(longitudes_alicia, bins=intervalos, density=True, alpha=0.6, label='Alice')
plt.hist(longitudes_oz, bins=intervalos, density=True, alpha=0.6, label='Oz')
plt.title('Distribución de la longitud de palabras')
plt.xlabel('numero de letras')
plt.ylabel('proporcion')
plt.legend()
plt.show()

plt.loglog(range(1, len(orden_alicia) + 1), orden_alicia, label='Alice')
plt.loglog(range(1, len(orden_oz) + 1), orden_oz, label='Oz')
plt.title('Distribución de rango y frecuencia')
plt.xlabel('posicion de la palabra')
plt.ylabel('frecuencia relativa')
plt.legend()
plt.show()

## bigramas y trigramas

Los hago por oración para no juntar palabras que en realidad estaban separadas

In [ ]:
conteo_bigramas_alicia = Counter()
conteo_trigramas_alicia = Counter()
for oracion in re.split(r'[.!?]+', texto_alicia.lower()):
    secuencia = re.findall(patron_palabras, oracion)
    secuencia = [palabra for palabra in secuencia if palabra not in palabras_extra]
    conteo_bigramas_alicia.update(zip(secuencia, secuencia[1:]))
    conteo_trigramas_alicia.update(zip(secuencia, secuencia[1:], secuencia[2:]))

conteo_bigramas_oz = Counter()
conteo_trigramas_oz = Counter()
for oracion in re.split(r'[.!?]+', texto_oz.lower()):
    secuencia = re.findall(patron_palabras, oracion)
    secuencia = [palabra for palabra in secuencia if palabra not in palabras_extra]
    conteo_bigramas_oz.update(zip(secuencia, secuencia[1:]))
    conteo_trigramas_oz.update(zip(secuencia, secuencia[1:], secuencia[2:]))

bigramas_alicia = conteo_bigramas_alicia.most_common(12)
bigramas_oz = conteo_bigramas_oz.most_common(12)
trigramas_alicia = conteo_trigramas_alicia.most_common(12)
trigramas_oz = conteo_trigramas_oz.most_common(12)

tabla_bigramas = pd.DataFrame({
    'bigrama alice': [' '.join(dato[0]) for dato in bigramas_alicia],
    'conteo alice': [dato[1] for dato in bigramas_alicia],
    'bigrama oz': [' '.join(dato[0]) for dato in bigramas_oz],
    'conteo oz': [dato[1] for dato in bigramas_oz]
})
tabla_bigramas

In [ ]:
tabla_trigramas = pd.DataFrame({
    'trigrama alice': [' '.join(dato[0]) for dato in trigramas_alicia],
    'conteo alice': [dato[1] for dato in trigramas_alicia],
    'trigrama oz': [' '.join(dato[0]) for dato in trigramas_oz],
    'conteo oz': [dato[1] for dato in trigramas_oz]
})
tabla_trigramas

## puntuación

Cuento también las comillas y guiones que aparecen en estas ediciones. Otra vez uso una proporción para poder comparar

In [ ]:
# quito los corchetes de las ilustraciones antes de contar
texto_puntuacion_alicia = re.sub(r'\[[^\]]*\]', ' ', texto_alicia).replace('_', '')
texto_puntuacion_oz = re.sub(r'\[[^\]]*\]', ' ', texto_oz).replace('_', '')
signos = ['.', ',', '?', '!', ';', ':', '"', "'", '“', '”', '—', '-']
puntuacion_alicia = Counter(simbolo for simbolo in texto_puntuacion_alicia if simbolo in signos)
puntuacion_oz = Counter(simbolo for simbolo in texto_puntuacion_oz if simbolo in signos)

tabla_puntuacion = pd.DataFrame({
    'signo': signos,
    'conteo alice': [puntuacion_alicia[signo] for signo in signos],
    'alice por 1000 caracteres': [round(puntuacion_alicia[signo] / len(texto_puntuacion_alicia) * 1000, 2) for signo in signos],
    'conteo oz': [puntuacion_oz[signo] for signo in signos],
    'oz por 1000 caracteres': [round(puntuacion_oz[signo] / len(texto_puntuacion_oz) * 1000, 2) for signo in signos]
})
tabla_puntuacion

In [ ]:
posiciones = np.arange(len(signos))
ancho = 0.38
tasas_alicia = [puntuacion_alicia[signo] / len(texto_puntuacion_alicia) * 1000 for signo in signos]
tasas_oz = [puntuacion_oz[signo] / len(texto_puntuacion_oz) * 1000 for signo in signos]

plt.figure(figsize=(11, 5))
plt.bar(posiciones - ancho / 2, tasas_alicia, width=ancho, label='Alice')
plt.bar(posiciones + ancho / 2, tasas_oz, width=ancho, label='Oz')
plt.xticks(posiciones, signos)
plt.ylabel('apariciones por cada 1,000 caracteres')
plt.title('Uso de signos de puntuación')
plt.legend()
plt.show()

## emojis

No espero encontrar emojis por la época de los libros, pero de todas formas los busco

In [ ]:
patron_emojis = re.compile(
    '[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u26FF\u2700-\u27BF]',
    flags=re.UNICODE
)
emojis_alicia = patron_emojis.findall(texto_alicia)
emojis_oz = patron_emojis.findall(texto_oz)

# alice y oz, en ese orden
Counter(emojis_alicia), Counter(emojis_oz)

## comparación

Junto los resultados y también veo cuántas de las 50 palabras principales se repiten en los dos libros

In [ ]:
principales_alicia = {dato[0] for dato in frecuencias_alicia.most_common(50)}
principales_oz = {dato[0] for dato in frecuencias_oz.most_common(50)}
coincidencia = len(principales_alicia & principales_oz) / len(principales_alicia | principales_oz)
signos_totales_alicia = sum(puntuacion_alicia.values())
signos_totales_oz = sum(puntuacion_oz.values())

comparacion = pd.DataFrame([
    ['palabras totales', len(palabras_alicia), len(palabras_oz)],
    ['tokens analizados', len(terminos_alicia), len(terminos_oz)],
    ['palabras unicas', len(set(terminos_alicia)), len(set(terminos_oz))],
    ['riqueza lexica', riqueza_alicia, riqueza_oz],
    ['longitud media', media_alicia, media_oz],
    ['signos por 1000 caracteres', round(signos_totales_alicia / len(texto_puntuacion_alicia) * 1000, 2),
     round(signos_totales_oz / len(texto_puntuacion_oz) * 1000, 2)],
    ['emojis', len(emojis_alicia), len(emojis_oz)]
], columns=['medida', 'alice', 'oz'])
comparacion

In [ ]:
obra_mas_larga = 'Alice' if len(palabras_alicia) > len(palabras_oz) else 'Oz'
obra_mas_diversa = 'Alice' if riqueza_alicia > riqueza_oz else 'Oz'
tasa_signos_alicia = signos_totales_alicia / len(texto_puntuacion_alicia) * 1000
tasa_signos_oz = signos_totales_oz / len(texto_puntuacion_oz) * 1000
obra_con_mas_puntuacion = 'Alice' if tasa_signos_alicia > tasa_signos_oz else 'Oz'
palabras_compartidas = sorted(principales_alicia & principales_oz)

# largo, riqueza, puntuación, coincidencia y algunas palabras compartidas
obra_mas_larga, obra_mas_diversa, obra_con_mas_puntuacion, round(coincidencia, 3), palabras_compartidas[:15]

## conclusión

Las palabras y los n-gramas dejan ver los personajes y lugares de cada historia. También hay vocabulario parecido porque los dos son libros infantiles de fantasía

No encontré emojis, lo cual tiene sentido por la época. La puntuación y las longitudes sí sirven para notar algunas diferencias entre los textos

Cosas a tomar en cuenta:

- solo usé un libro de cada autor
- los resultados cambian dependiendo de las palabras que quite
- algunas comillas o guiones pueden venir de la edición de Gutenberg
- no junté variantes de una misma palabra